# 06 · Mean center, checked against pyproj

## What this checks

**Mean center** finds the average position of a set of points. This check covers release 0.5.0.

Fieldwork says how it does this. It converts each point to UTM, a flat grid in metres. It averages the eastings and
the northings with equal weight. Then it converts the average back to longitude and latitude.

A centre that is in the wrong place moves every distance measured from it. So it is worth checking.

## How it is checked

We repeat the same steps with **pyproj**, the Python link to PROJ. PROJ is the projection library most GIS software
uses. Fieldwork uses a different one in the browser, proj4js. The two share no code.

The data are six points. One of them is far from the others, on purpose.

## What should happen

1. The average easting and northing agree within 1 millimetre.
2. The returned position agrees within 1 millimetre, measured as a distance on the ground. We measure a distance
   because a difference in degrees is not a distance.
3. The number of points is the same on both sides.

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_06_mean_center import run

report = run(Path.cwd() / "fixtures" / "mean-center-001.json")
print("criterion:", report["criterion"])
print("agrees:", report["agrees"], "over", report["points"], "points")
print(f"easting  {report['eastingDifferenceM']:.3e} m")
print(f"northing {report['northingDifferenceM']:.3e} m")
print(f"ground separation {report['separationM']:.3e} m")


## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.5.0:
**Agrees**.

The eastings match exactly. The northings differ by about 9 × 10⁻¹⁰ m. The two positions are about 4 × 10⁻¹⁰ m
apart on the ground. Both are far below a millimetre, and far below any accuracy a position could have. Both sides counted 6
points.

So proj4js and PROJ agree on this conversion, and the average has no weights, as Fieldwork says.

Running the cell above repeats the check on the same data.

## The point that pulls the centre

The cell below shows how far each point is from the centre. One point is about 25 km from the rest. It pulls the
centre towards it, on both sides equally.

In [ ]:
# The outlier is the point of the fixture: an unweighted mean is pulled, and both
# implementations are pulled identically. Agreement is not a statement about the statistic.
import pyproj
geod = pyproj.Geod(ellps="WGS84")
fixture = json.loads((Path.cwd() / "fixtures" / "mean-center-001.json").read_text())
centre = report["fieldwork"]["coordinates"]
for point in fixture["input"]["points"]:
    *_ , distance = geod.inv(centre[0], centre[1], point["longitude"], point["latitude"])
    print(f"{point['id']:<4} {distance/1000:7.2f} km from the mean centre")
print()
print(report["notEstablished"])


## What this does not show

- That a mean centre is a good summary of these points. The far point pulls it, and both sides are pulled the same
  way. That is agreement about arithmetic, not about the method.